# Public data experiments
This is the notebook to run the public data experiment

In [1]:
import sys
from pathlib import Path

# Add the workspace root to sys.path
workspace_root = Path("/Users/eduardohome/workspace/benchmark_credit_scoring")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Now you can import from src
from src.utils.csv_config import read_csv_by_name, create_imbalanced_datasets
from src.model.functions import save_model
try:
    from src.model.functions import get_train_model
except Exception as exc:
    # provide helpful message if import fails
    print("Failed to import model utilities:", exc)
    print("This often means a missing dependency such as libomp for LightGBM.")
    print("On macOS you can install it with `brew install libomp` and then restart the kernel.")
    raise

# General imports
import pandas as pd

In [2]:
# -------------------------
# Settings
# -------------------------
data_dir = Path("/Users/eduardohome/workspace/benchmark_credit_scoring/data")
dataset_name = "german"
target_col = "target"

# Edit this list whenever you want more/less imbalance datasets
imbalance_levels = [3, 5, 10, 15, 20]

# list of estimators understood by functions.get_model
model_choices = ["RandomForest", "XGBoost", "LightGBM", "BoostDT", "BoostSGB",
                 "BagNN", "AdaBoost-DT", "SVM-Rbf", "ANN"]
product = "default"

In [3]:
dataset_path = data_dir / "input" / f"{dataset_name}.csv"

# Check if file exists and read it using the helper from src.csv_config
if dataset_path.exists():
    # Use the dataset-specific defaults (german uses tab separator)
    df = read_csv_by_name(str(dataset_path), dataset_name)
    df.head()
else:
    print(f"File not found: {dataset_path}")

In [4]:
create_imbalanced_datasets(
    dataset_name=dataset_name,
    data_dir=data_dir,
    imbalance_levels=imbalance_levels,  # <-- modify freely
    target_col="target",
)

Original dataset: german
Total samples: 1000
Positive samples: 300
Negative samples: 700
Positive rate: 0.3000

german_3.csv | Total=721 | Positives=21 | Rate=0.0291
german_5.csv | Total=736 | Positives=36 | Rate=0.0489
german_10.csv | Total=777 | Positives=77 | Rate=0.0991
german_15.csv | Total=823 | Positives=123 | Rate=0.1495
german_20.csv | Total=875 | Positives=175 | Rate=0.2000

Finished generating imbalanced datasets.


In [5]:
from pathlib import Path
import pandas as pd

# -------------------------
# Build list of dataset variants to run
# (includes original + the imbalanced ones)
# -------------------------
dataset_variants = [dataset_name] + [f"{dataset_name}_{pct}" for pct in imbalance_levels]

results = []  # collect stats

for dataset_variant in dataset_variants:
    dataset_path = data_dir / "input" / f"{dataset_variant}.csv"

    if not dataset_path.exists():
        print(f"[SKIP] File not found: {dataset_path}")
        continue

    # Read each dataset variant
    df = read_csv_by_name(str(dataset_path), dataset_name)  # keep base name for separator config etc.

    # Prepare column lists for this df (safe even if row counts differ)
    feature_cols     = df.columns.tolist()
    numerical_cols   = df.select_dtypes(include="number").columns.tolist()
    categorical_cols = df.select_dtypes(include="object").columns.tolist()
    ids_cols         = []
    target_cols      = [target_col]
    exclude_cols     = []

    # Quick dataset stats
    n_total = len(df)
    n_pos = int(df[target_col].sum())
    pos_rate = float(df[target_col].mean())

    print("\n" + "=" * 80)
    print(f"DATASET: {dataset_variant} | n={n_total} | pos={n_pos} | pos_rate={pos_rate:.4f}")
    print("=" * 80)

    for model_choice in model_choices:
        print(f"\n=== training {model_choice} on {dataset_variant} ===")

        pipeline, best_params, best_score = get_train_model(
            data=df,
            feature_cols=feature_cols,
            numerical_cols=numerical_cols,
            categorical_cols=categorical_cols,
            ids_cols=ids_cols,
            target_cols=target_cols,
            exclude_cols=exclude_cols,
            product=product,
            algorithm=model_choice,
            test_mode=True,  # set False for full search
        )

        print("Best hyperparameters:", best_params)
        print("Best CV score:", best_score)

        # Save the trained model object
        model_path = data_dir / "output" / f"{dataset_variant}_{model_choice}.pkl"
        save_model(str(model_path), pipeline, environment="local")
        print(f"trained pipeline saved to {model_path}")

        results.append(
            {
                "dataset": dataset_variant,
                "model": model_choice,
                "n_total": n_total,
                "n_pos": n_pos,
                "pos_rate": pos_rate,
                "params": best_params,
                "score": best_score,
            }
        )

# optional: look at all results afterwards
results_df = pd.DataFrame(results)
results_df


DATASET: german | n=1000 | pos=300 | pos_rate=0.3000

=== training RandomForest on german ===
Search space for algorithm=RandomForest, product=default, test_mode=True: {'model__n_estimators': Integer(low=50, high=300, prior='uniform', transform='identity'), 'model__max_depth': Integer(low=1, high=50, prior='uniform', transform='identity'), 'model__min_samples_split': Integer(low=2, high=20, prior='uniform', transform='identity'), 'model__min_samples_leaf': Integer(low=1, high=10, prior='uniform', transform='identity'), 'model__max_features': Categorical(categories=('sqrt',), prior=None), 'model__bootstrap': Categorical(categories=(True, False), prior=None), 'model__class_weight': Categorical(categories=('balanced', 'balanced_subsample', None), prior=None)}
Best hyperparameters: OrderedDict([('model__bootstrap', False), ('model__class_weight', 'balanced_subsample'), ('model__max_depth', 5), ('model__max_features', 'sqrt'), ('model__min_samples_leaf', 7), ('model__min_samples_split', 4)

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer